In [1]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [2]:
import warnings
warnings.filterwarnings('ignore')

### Verify GPU

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_type = torch.float32 if device.type == "cuda" else torch.float16
device, torch_type

(device(type='cuda'), torch.float32)

### Loading package

In [4]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [5]:
from py.utils import verifyDir,verifyFile

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

DATA_PATH = os.getenv('DATA_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
DATA_PATH, MODEL_PATH

('/media/felipe/DATA19/datasets/', '/media/felipe/DATA19/models/')

In [7]:
MODEL_NAME="OneFormer_Swin_Large"
SEG_DATASET="ade20k" # cityscapes

In [8]:
ADE20K_DIR = f"{DATA_PATH}{SEG_DATASET}/"
QSCORE_PATH=f"{DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{DATA_PATH}pp2/images/"
SEGMENT_DIR = f"{DATA_PATH}pp2/segmentations/{SEG_DATASET}/{MODEL_NAME}/"

In [9]:
verifyDir(SEGMENT_DIR)

### Loading data

In [10]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
cities = np.sort(data_df["city"].unique()).tolist()
data_df["image_path"] = f"{IMAGES_PATH}" + data_df["image_path"]
data_df

CPU times: user 143 ms, sys: 37.2 ms, total: 180 ms
Wall time: 180 ms


,image_id,lat,long,city,country,continent,safety,beautiful,wealthy,lively,boring,depressing,image_path
0,50e5f7d4d7c3df413b00056a,22.310524,114.170637,Hong Kong,China,Asia,4.135536,1.574074,2.962963,4.199346,5.000000,0.000000,/media/felipe/DATA19/datasets/pp2/images/Hong ...
1,50e5f7d4d7c3df413b00056b,22.274799,114.192828,Hong Kong,China,Asia,3.560981,2.229437,5.277778,5.662393,7.777778,3.333333,/media/felipe/DATA19/datasets/pp2/images/Hong ...
2,50e5f7d4d7c3df413b00056c,22.291117,114.147373,Hong Kong,China,Asia,4.514946,3.333333,3.333333,4.746693,3.611111,0.000000,/media/felipe/DATA19/datasets/pp2/images/Hong ...
3,50e5f7d4d7c3df413b00056d,22.314273,114.177176,Hong Kong,China,Asia,4.852448,3.333333,5.083333,3.333333,8.333333,3.327381,/media/felipe/DATA19/datasets/pp2/images/Hong ...
4,50e5f7d4d7c3df413b00056e,22.332412,114.204790,Hong Kong,China,Asia,4.975207,2.129630,3.680556,4.343857,2.500000,4.444444,/media/felipe/DATA19/datasets/pp2/images/Hong ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
108433,5185d533fdc9f03fd500148d,45.635489,-73.514755,Montreal,Canada,North America,6.111111,5.833333,2.333333,3.495058,3.333333,9.166667,/media/felipe/DATA19/datasets/pp2/images/Montr...
108434,5185d533fdc9f03fd500148e,45.581265,-73.511511,Montreal,Canada,North America,4.797811,3.251984,2.777778,2.148291,7.222222,7.777778,/media/felipe/DATA19/datasets/pp2/images/Montr...
108435,5185d534fdc9f03fd500148f,45.591299,-73.520681,Montreal,Canada,North America,5.006830,6.888889,7.777778,6.176471,5.833333,2.611111,/media/felipe/DATA19/datasets/pp2/images/Montr...
108436,5185d534fdc9f03fd5001490,45.470656,-73.631901,Montreal,Canada,North America,4.551622,3.333333,5.000000,5.008547,3.611111,2.962963,/media/felipe/DATA19/datasets/pp2/images/Montr...


##### Convert classes and colors file

In [11]:
from py.datasets import UrbanPhysicalDisorder

uss = UrbanPhysicalDisorder(data_path=DATA_PATH)
uss.generate_dataset(dataset="ade20k")

objects_df = uss.get_urban_street_categories()
objects_df

,class_id,main_class,class_name,RGB_color,hex_color,isthing,group_name
0,1,wall,wall,"(120, 120, 120)",#787878,0,construction
1,2,building,building;edifice,"(180, 120, 120)",#B47878,0,construction
2,3,sky,sky,"(6, 230, 230)",#06E6E6,0,sky
3,4,floor,floor;flooring,"(80, 50, 50)",#503232,0,floor
4,5,tree,tree,"(4, 200, 3)",#04C803,0,vegetation
...,...,...,...,...,...,...,...
145,146,shower,shower,"(0, 133, 255)",#0085FF,0,indoor_object
146,147,radiator,radiator,"(255, 214, 0)",#FFD600,1,terrain_vehicle_related
147,148,glass,glass;drinking;glass,"(25, 194, 194)",#19C2C2,1,miscellaneous
148,149,clock,clock,"(102, 255, 0)",#66FF00,1,miscellaneous


### Zero-shot Segmenter

In [12]:
from py.models.segmentation import ImageSegmenter

ts = ImageSegmenter(model_name=MODEL_NAME, model_path=MODEL_PATH)
ts.to_device(device)
ts.eval()
ts.model_zoo()
ts.print_trainable_parameters()
ts.get_model()

Loading weights:   0%|          | 0/826 [00:00<?, ?it/s]

Model zoo ADE20K and CityScapes: ['OneFormer_Swin_Large', 'OneFormer_Swin_Tiny', 'OneFormer_Dinat_Large', 'Mask2Former_Swin_Large', 'Mask2Former_Swin_Base', 'Mask2Former_Swin_Small', 'Mask2Former_Swin_Tiny', 'SegFormer_B0', 'SegFormer_B1', 'SegFormer_B2', 'SegFormer_B3', 'SegFormer_B4', 'SegFormer_B5'] 

Model Zoo only ADE20K-pytorch: PSP_ResNet50_Dilated 

Model Zoo only ADE20K-keras: DeepLabV3_Exception65 

Model arch used: shi-labs/oneformer_ade20k_swin_large 

Trainable parameters in the model:
   Total parameters: 218,811,148
   Trainable parameters: 218,811,148
   Trainable parameters percentage: 100.0000%


OneFormer_Swin(
  (model): OneFormerForUniversalSegmentation(
    (model): OneFormerModel(
      (pixel_level_module): OneFormerPixelLevelModule(
        (encoder): SwinBackbone(
          (embeddings): SwinEmbeddings(
            (patch_embeddings): SwinPatchEmbeddings(
              (projection): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
            )
            (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (encoder): SwinEncoder(
            (layers): ModuleList(
              (0): SwinStage(
                (blocks): ModuleList(
                  (0): SwinLayer(
                    (layernorm_before): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
                    (attention): SwinAttention(
                      (self): SwinSelfAttention(
                        (query): Linear(in_features=192, out_features=192, bias=True)
                        (key): Linear(in_features=

In [13]:
columns_to_keep = ["image_id", "seg_image_path", "seg_overlay_image_path", "mask_path", "ratio_path"]

In [14]:
%%time
segment_df = pd.DataFrame()
for city in cities:
    print("Evaluating city", city)
    OUT_DIR = f"{SEGMENT_DIR}/{city}/"
    verifyDir(OUT_DIR)
    verifyDir(f"{OUT_DIR}/masks/")
    verifyDir(f"{OUT_DIR}/ratios/")
    verifyDir(f"{OUT_DIR}/segmented_images/")
    verifyDir(f"{OUT_DIR}/segmented_images_overlay/")

    city_df = data_df[ (data_df["city"]==city) ][["image_id", "image_path"]].copy()
    country = data_df[ (data_df["city"]==city) ]["country"].unique()[0]
    city_df.reset_index(drop=True, inplace=True)
    city_segment_df = pd.DataFrame()

    for index, row in tqdm(city_df.iterrows()):
        img_path = row["image_path"]
        out_name = img_path.split("/")[-1].replace(".JPG", "")

        if verifyFile(f"{OUT_DIR}/masks/{out_name}.pkl") and verifyFile(f"{OUT_DIR}/ratios/{out_name}.csv") and verifyFile(f"{OUT_DIR}/segmented_images/{out_name}.png") and verifyFile(f"{OUT_DIR}/segmented_images_overlay/{out_name}.png"):
            ratio_df = pd.read_csv(f"{OUT_DIR}/ratios/{out_name}.csv", sep=";", low_memory=False)
        else:
            image = Image.open(img_path).convert("RGB")

            ratio_df, masks, seg_img, seg_overlay_img = ts.zeroshot_segmentation(image, objects_df, alpha=0.6)

            # saving image
            seg_img.save(f"{OUT_DIR}/segmented_images/{out_name}.png")
            seg_overlay_img.save(f"{OUT_DIR}/segmented_images_overlay/{out_name}.png")
            #cv2.imwrite(output_seg, np.array(seg_img))
            #cv2.imwrite(output_seg_overlay, np.array(seg_overlay_img))

            # masks
            joblib.dump(masks, f"{OUT_DIR}/masks/{out_name}.pkl")

            # ratios
            ratio_df.to_csv(f"{OUT_DIR}/ratios/{out_name}.csv", sep=";", index=False)

        df_pivot = uss.parse_ratios(out_name, ratio_df)
        df_pivot["seg_image_path"] = f"{city}/segmented_images/{out_name}.png"
        df_pivot["seg_overlay_image_path"] = f"{city}/segmented_images_overlay/{out_name}.png"
        df_pivot["mask_path"] = f"{city}/masks/{out_name}.pkl"
        df_pivot["ratio_path"] = f"{city}/ratios/{out_name}.csv"
        
        city_segment_df = pd.concat([city_segment_df, df_pivot], ignore_index=True)
        city_segment_df = city_segment_df[columns_to_keep + [col for col in city_segment_df.columns if col not in columns_to_keep]].copy()
        city_segment_df.fillna(0, inplace=True)

    city_df = pd.merge(city_df, city_segment_df, on="image_id", how="inner").copy()
    city_df.fillna(0, inplace=True)
    city_df.drop(columns=["image_path"], inplace=True)
    city_df.to_csv(f"{OUT_DIR}/segmentations.csv", sep=";", index=False)
        
    segment_df = pd.concat([segment_df, city_df], ignore_index=True)
    segment_df.fillna(0, inplace=True)

Evaluating city Amsterdam


622it [02:11,  4.73it/s]


Evaluating city Atlanta


3942it [13:59,  4.70it/s]


Evaluating city Bangkok


...

Evaluating city Washington DC


928it [04:43,  3.28it/s]


Evaluating city Zagreb


1052it [06:43,  2.61it/s]

CPU times: user 6h 8min 7s, sys: 12min 49s, total: 6h 20min 57s
Wall time: 10h 18min 26s


In [15]:
segment_df

,image_id,seg_image_path,seg_overlay_image_path,mask_path,ratio_path,bicycle,building,car,flowerpot,person,...,crt_screen,monitor,seat,bar,sofa_pillow,vitrine,billiard,chandelier,microwave,stove
0,513d564efdc9f03587002f73,Amsterdam/segmented_images/513d564efdc9f035870...,Amsterdam/segmented_images_overlay/513d564efdc...,Amsterdam/masks/513d564efdc9f03587002f73.pkl,Amsterdam/ratios/513d564efdc9f03587002f73.csv,1.651667,30.939167,0.421667,0.8825,0.364167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,513d564ffdc9f03587002f74,Amsterdam/segmented_images/513d564ffdc9f035870...,Amsterdam/segmented_images_overlay/513d564ffdc...,Amsterdam/masks/513d564ffdc9f03587002f74.pkl,Amsterdam/ratios/513d564ffdc9f03587002f74.csv,0.000000,14.285000,13.425833,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,513d564ffdc9f03587002f75,Amsterdam/segmented_images/513d564ffdc9f035870...,Amsterdam/segmented_images_overlay/513d564ffdc...,Amsterdam/masks/513d564ffdc9f03587002f75.pkl,Amsterdam/ratios/513d564ffdc9f03587002f75.csv,0.000000,0.000000,0.000000,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,513d5650fdc9f03587002f76,Amsterdam/segmented_images/513d5650fdc9f035870...,Amsterdam/segmented_images_overlay/513d5650fdc...,Amsterdam/masks/513d5650fdc9f03587002f76.pkl,Amsterdam/ratios/513d5650fdc9f03587002f76.csv,0.000000,0.240000,0.203333,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,513d5651fdc9f03587002f77,Amsterdam/segmented_images/513d5651fdc9f035870...,Amsterdam/segmented_images_overlay/513d5651fdc...,Amsterdam/masks/513d5651fdc9f03587002f77.pkl,Amsterdam/ratios/513d5651fdc9f03587002f77.csv,0.000000,29.627500,0.000000,0.0000,0.072500,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108433,5140ba43fdc9f04926001f98,Zagreb/segmented_images/5140ba43fdc9f04926001f...,Zagreb/segmented_images_overlay/5140ba43fdc9f0...,Zagreb/masks/5140ba43fdc9f04926001f98.pkl,Zagreb/ratios/5140ba43fdc9f04926001f98.csv,0.000000,4.878333,0.057500,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
108434,5140ba45fdc9f04926001f99,Zagreb/segmented_images/5140ba45fdc9f04926001f...,Zagreb/segmented_images_overlay/5140ba45fdc9f0...,Zagreb/masks/5140ba45fdc9f04926001f99.pkl,Zagreb/ratios/5140ba45fdc9f04926001f99.csv,0.000000,0.000000,0.135000,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
108435,5140ba46fdc9f04926001f9a,Zagreb/segmented_images/5140ba46fdc9f04926001f...,Zagreb/segmented_images_overlay/5140ba46fdc9f0...,Zagreb/masks/5140ba46fdc9f04926001f9a.pkl,Zagreb/ratios/5140ba46fdc9f04926001f9a.csv,0.000000,28.957500,0.093333,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
108436,5140ba47fdc9f04926001f9b,Zagreb/segmented_images/5140ba47fdc9f04926001f...,Zagreb/segmented_images_overlay/5140ba47fdc9f0...,Zagreb/masks/5140ba47fdc9f04926001f9b.pkl,Zagreb/ratios/5140ba47fdc9f04926001f9b.csv,0.000000,0.865000,3.667500,0.0000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
np.sort(segment_df.columns)

array(['airplane', 'animal', 'armchair', 'ashcan', 'awning', 'bag',
       'ball', 'bar', 'barrel', 'basket', 'bed', 'bench', 'bicycle',
       'billiard', 'blind', 'boat', 'book', 'bottle', 'box', 'bridge',
       'building', 'bulletin', 'bus', 'cabinet', 'canopy', 'car',
       'carpet', 'ceiling', 'chair', 'chandelier', 'clock', 'clothes',
       'column', 'conveyor', 'counter', 'crt_screen', 'curtain', 'door',
       'escalator', 'fan', 'fence', 'field', 'flag', 'floor', 'flower',
       'flowerpot', 'food', 'fountain', 'glass', 'grandstand', 'grass',
       'ground', 'handrail', 'hill', 'house', 'image_id', 'lake', 'lamp',
       'land', 'light', 'mask_path', 'microwave', 'mirror', 'monitor',
       'motorcycle', 'mountain', 'painting', 'palm', 'path', 'pedestal',
       'person', 'pier', 'plant', 'plaything', 'pole', 'poster',
       'railing', 'ratio_path', 'refrigerator', 'river', 'road', 'rock',
       'runway', 'sand', 'sconce', 'screen_projection', 'sculpture',
       'sea',

In [17]:
segment_df.to_csv(f"{SEGMENT_DIR}/segmentations.csv", sep=";", index=False)